# 스프린트미션16 4팀_김명환

In [1]:
import importlib
import sys
import subprocess

def install_if_missing(package_name, module_name=None, index_url=None):
    # module_name이 없으면 package_name을 그대로 사용
    module_name = module_name or package_name.replace("-", "_")

    if importlib.util.find_spec(module_name) is None:
        print(f"{package_name} 설치 중...")
        cmd = [sys.executable, "-m", "pip", "install"]
        if index_url:
            cmd += ["--index-url", index_url]
        cmd.append(package_name)
        subprocess.check_call(cmd)
    else:
        print(f"{package_name} 이미 설치되어 있음.")

# 사용 예시
install_if_missing("helper-plot-hangul", "helper_plot_hangul", "https://test.pypi.org/simple/")
install_if_missing("helper-utils", "helper_utils", "https://test.pypi.org/simple/")


helper-plot-hangul 이미 설치되어 있음.
helper-utils 이미 설치되어 있음.


In [2]:
# import importlib
# from helper_plot_hangul import helper_plot_hangul
# importlib.reload(helper_plot_hangul)

# import helper_utils.helper_logger as helper_logger
# importlib.reload(helper_logger)

# import helper_utils.helper_utils_colab as helper_utils_colab
# importlib.reload(helper_utils_colab)

from helper_plot_hangul import *
from helper_utils.helper_logger import *
from helper_utils.helper_utils_colab import *
from helper_utils.helper_utils_print import *
from helper_utils.helper_pandas import *


In [3]:
# 기본 라이브러리

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.metrics import average_precision_score

# --- 기타 라이브러리 ---
from PIL import Image
from PIL import ImageFilter
from PIL import ImageDraw
import albumentations as A
import IPython.display
#from tqdm import tqdm
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# --- PyTorch: 딥러닝 관련 ---
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.datasets import CocoDetection
from torchvision.transforms import functional as TF
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- 기타 ---
import re
import os
import sys
import copy
import json
import math
import random
import yaml
import shutil
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from datetime import datetime
from datetime import timezone, timedelta
import pytz
__kst = pytz.timezone('Asia/Seoul')

# GPU 설정
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

  # 재현 가능한 결과를 위해
np.random.seed(42)
torch.manual_seed(42)
if __device == 'cuda':
    torch.cuda.manual_seed_all(42)

logger.info(f"라이브러리 로드 완료 사용장치:{__device}")

2025-12-07 23:11:00 I [helper_pandas:74] - 라이브러리 로드 완료 사용장치:cpu


### > 설정 < 플레그

In [4]:
DEBUG_ON = False if IS_COLAB else True
DEBUG_ON = False
TRAIN_ON = False
logger.info(f"IS_COLAB={IS_COLAB}")
logger.info(f"DEBUG_ON={DEBUG_ON}")


2025-12-07 23:11:00 I [helper_pandas:4] - IS_COLAB=False
2025-12-07 23:11:00 I [helper_pandas:5] - DEBUG_ON=False


In [5]:
from ultralytics import YOLO
# GPU 확인
device = __device
logger.info(f"사용 디바이스: {device}")
logger.info(f"CUDA 버전: {torch.version.cuda}")

2025-12-07 23:11:00 I [helper_pandas:4] - 사용 디바이스: cpu
2025-12-07 23:11:00 I [helper_pandas:5] - CUDA 버전: None


In [6]:
os.environ['YOLO_VERBOSE'] = 'False'
os.environ['ULTRALYTICS_LOG_LEVEL'] = 'WARNING'  # 또는 'ERROR'

In [7]:
import os, sys
import importlib
sys.path.insert(0, os.getcwd())

# 기존 모듈 완전 제거
if 'yolo_eval' in sys.modules:
    del sys.modules['yolo_eval']
    
# 하위 모듈도 제거
for key in list(sys.modules.keys()):
    if key.startswith('yolo_eval.'):
        del sys.modules[key]

# 새로 임포트
from yolo_eval import *

logger.info("YOLOEvaluator 클래스 로드 완료")

2025-12-07 23:11:00 I [helper_logger:58] - EvaluationMetrics 클래스 로드 완료
2025-12-07 23:11:00 I [helper_logger:24] - PredictionResult 클래스 로드 완료
2025-12-07 23:11:00 I [helper_logger:637] - QuantizedModelWrapper 모듈 로드 완료
2025-12-07 23:11:00 I [helper_logger:449] - YOLOEvaluator 클래스 로드 완료
2025-12-07 23:11:01 I [helper_logger:255] - YOLOEvaluationPipeline 클래스 로드 완료
2025-12-07 23:11:01 I [helper_pandas:17] - YOLOEvaluator 클래스 로드 완료


In [8]:
yolov8m_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005')
yolov8m_result_path = my_driver_path('modeling', 'model', 'modeling16', 'yolov8m_20251205_2005', 'results')
yolov8m_best_path = my_driver_path(yolov8m_path, 'weights', 'best.pt', create=False)
yolov8m_2class_yaml_path = my_driver_path(yolov8m_path, 'weights', 'mission_16_yolo.yaml', create=False)

logger.setLevel(logging.INFO)
yolo_dataset_path = my_cache_path("yolo", "the-oxfordiiit-pet-dataset")
yaml_path, train_df, valid_df, test_df, validation_results = oxfordiit_pet_to_yolo(max_samples_per_split=(30, 20, 10),
                                                                                   label_mode="species",
                                                                                   output_dir=Path(yolo_dataset_path))
logger.setLevel(logging.DEBUG)

logger.info(f"yaml_path: {yaml_path}")
logger.info(f"yolov8m_path: {yolov8m_path}")
logger.info(f"yolov8m_result_path: {yolov8m_result_path}")

logger.info(f"yolov8m_best_path: {yolov8m_best_path}")
logger.info(f"yolov8m_2class_yaml_path: {yolov8m_2class_yaml_path}")

logger.info("="*80)
logger.info("YOLO 모델 ONNX 내보내기 (FP32 + FP16 양자화)")
logger.info("="*80)
logger.info("미션 요구사항:")
logger.info("  1. ONNX 내보내기 - ONNX FP32 형식")
logger.info("  2. 양자화 - ONNX FP16 (half-precision)")
logger.info("")

# 원본 모델 로드
model = YOLO(yolov8m_best_path)


2025-12-07 23:11:01 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-07 23:11:01 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\results


Processing test: 100%|██████████| 10/10 [00:00<00:00, 5218.09it/s]
2025-12-07 23:11:01,883 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Abyssinian_104.txt
2025-12-07 23:11:01,920 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Bengal_111.txt
2025-12-07 23:11:01,927 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Bengal_175.txt
2025-12-07 23:11:01,947 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Egyptian_Mau_14.txt
2025-12-07 23:11:01,960 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Egyptian_Mau_156.txt
2025-12-07 23:11:01,964 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxf

2025-12-07 23:11:02 I [helper_pandas:13] - yaml_path: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\data.yaml
2025-12-07 23:11:02 I [helper_pandas:14] - yolov8m_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005
2025-12-07 23:11:02 I [helper_pandas:15] - yolov8m_result_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\results
2025-12-07 23:11:02 I [helper_pandas:17] - yolov8m_best_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt
2025-12-07 23:11:02 I [helper_pandas:18] - yolov8m_2class_yaml_path: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\mission_16_yolo.yaml
2025-12-07 23:11:02 I [helper_pandas:20] - ================================================================================
2025-12-07 23:11:02 I [helper_pandas:21] - YOLO 모델 ONNX 내보내기 (FP32 + FP16 양자화)
2025-12-07 23:11:02 I [helper_pandas:22] - ================================================================================


In [9]:
# ============================================================================
# 1. ONNX FP32 내보내기 (Ultralytics 기본 지원)
# ============================================================================
logger.info("[1/2] ONNX FP32 모델 내보내기 중...")
output_onnx_fp32_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp32', create=True)

onnx_fp32_export = model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    dynamic=True,
)

logger.info(f"ONNX FP32 내보내기 완료: {onnx_fp32_export}")

# 지정 위치로 복사
onnx_fp32_final = os.path.join(output_onnx_fp32_path, 'yolov8m_fp32.onnx')
if os.path.exists(onnx_fp32_export) and onnx_fp32_export != onnx_fp32_final:
    shutil.copy(onnx_fp32_export, onnx_fp32_final)
    logger.info(f"복사 완료: {onnx_fp32_final}")


2025-12-07 23:11:02 I [helper_pandas:4] - [1/2] ONNX FP32 모델 내보내기 중...
2025-12-07 23:11:02 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32
Ultralytics 8.3.235  Python-3.10.19 torch-2.5.1+cpu CPU (12th Gen Intel Core(TM) i7-1260P)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (49.6 MB)

ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.78...
ONNX: export success  6.2s, saved as 'D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.onnx' (98.7 MB)

Export complete (8.0s)
Results saved to D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights
Predict:         yolo predict task=detect model=D:\GoogleDrive\modeling\model\model

In [10]:
# ============================================================================
# 2. ONNX FP16 내보내기 (half-precision 양자화)
# ============================================================================
logger.info("[2/2] ONNX FP16 모델 내보내기 중...")
output_onnx_fp16_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_fp16', create=True)

onnx_fp16_export = model.export(
    format='onnx',
    half=True,      # FP16 half-precision 양자화
    imgsz=640,
    simplify=True,
    dynamic=True,
)
logger.info(f"ONNX FP16 내보내기 완료: {onnx_fp16_export}")

# 지정 위치로 복사
onnx_fp16_final = os.path.join(output_onnx_fp16_path, 'yolov8m_fp16.onnx')
if os.path.exists(onnx_fp16_export) and onnx_fp16_export != onnx_fp16_final:
    shutil.copy(onnx_fp16_export, onnx_fp16_final)
    logger.info(f"복사 완료: {onnx_fp16_final}")



2025-12-07 23:11:10 I [helper_pandas:4] - [2/2] ONNX FP16 모델 내보내기 중...
2025-12-07 23:11:10 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp16
Ultralytics 8.3.235  Python-3.10.19 torch-2.5.1+cpu CPU (12th Gen Intel Core(TM) i7-1260P)
WARNING half=True only compatible with GPU export, i.e. use device=0, setting half=False.
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (49.6 MB)

ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.78...
ONNX: export success  7.7s, saved as 'D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.onnx' (98.7 MB)

Export complete (9.4s)
Results saved to D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\w

In [11]:
# ============================================================================
# 3. INT8 양자화는 ONNX 형식에서 지원되지 않습니다
# ============================================================================
# ONNX INT8 시도 결과: AssertionError - 'int8' is not supported for format='onnx'
#
# INT8 양자화를 원하시면 다음 대안을 사용하세요:
# 1. TFLite INT8 (모바일/임베디드)
# 2. OpenVINO INT8 (Intel CPU 최적화)
# 3. TensorRT INT8 (NVIDIA GPU)
#
# 참고:
# - Ultralytics GitHub Issue #21391
# - ONNX는 FP32, FP16만 지원
# - CPU 환경에서는 FP16도 GPU만큼 효과적이지 않음
# logger.info("[참고] ONNX INT8은 Ultralytics에서 지원하지 않습니다.")
# logger.info("대안: TFLite, OpenVINO, TensorRT 사용 가능")

In [12]:
# ============================================================================
# 4. OpenVINO INT8 양자화 (Intel CPU 최적화)
# ============================================================================
logger.info("[3/3] OpenVINO INT8 모델 내보내기 중...")
logger.info("OpenVINO는 Intel CPU에 최적화된 INT8 양자화를 제공합니다")

# try:
# OpenVINO INT8 export
output_openvino_int8_path = model.export(
    format='openvino',
    int8=True,      # INT8 양자화
    imgsz=640,
#        dynamic=True,
)

logger.info(f"OpenVINO INT8 내보내기 완료: {output_openvino_int8_path}")

# 결과 확인
if os.path.exists(output_openvino_int8_path):
    openvino_files = os.listdir(output_openvino_int8_path) if os.path.isdir(output_openvino_int8_path) else [output_openvino_int8_path]
    logger.info(f"생성된 파일: {openvino_files}")
    
    # 크기 계산
    if os.path.isdir(output_openvino_int8_path):
        total_size = sum(os.path.getsize(os.path.join(output_openvino_int8_path, f)) 
                        for f in openvino_files if os.path.isfile(os.path.join(output_openvino_int8_path, f)))
    else:
        total_size = os.path.getsize(output_openvino_int8_path)
    
    openvino_int8_size = total_size / (1024 * 1024)
    logger.info(f"OpenVINO INT8 모델 크기: {openvino_int8_size:.2f} MB")

# except Exception as e:
#     logger.error(f"OpenVINO INT8 내보내기 실패: {e}")
#     logger.error("OpenVINO 라이브러리가 설치되지 않았을 수 있습니다.")
#     logger.error("설치 방법: pip install openvino-dev")
#     openvino_int8_export = None
#     openvino_int8_size = None

2025-12-07 23:11:20 I [helper_pandas:4] - [3/3] OpenVINO INT8 모델 내보내기 중...
2025-12-07 23:11:20 I [helper_pandas:5] - OpenVINO는 Intel CPU에 최적화된 INT8 양자화를 제공합니다
Ultralytics 8.3.235  Python-3.10.19 torch-2.5.1+cpu CPU (12th Gen Intel Core(TM) i7-1260P)
WARNING INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from 'D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (49.6 MB)

OpenVINO: starting export with openvino 2025.4.0-20398-7a975177ff4-releases/2025/4...
OpenVINO: collecting INT8 calibration images from 'data=coco8.yaml'
Fast image access  (ping: 0.10.1 ms, read: 217.498.5 MB/s, size: 54.0 KB)
Scanning D:\GoogleDrive\homepage\스프린트미션\스프린트미션_작업중\datasets\coco8\labels\val.cache... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 4.0Kit/s

Output()

Output()

OpenVINO: export success  60.4s, saved as 'D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model\' (25.4 MB)

Export complete (62.8s)
Results saved to D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights
Predict:         yolo predict task=detect model=D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model imgsz=640 int8 
Validate:        yolo val task=detect model=D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model imgsz=640 data=/home/spai0433/work/mission16/cache_local/yolo/the-oxfordiiit-pet-dataset/dataset.yaml int8 
Visualize:       https://netron.app
2025-12-07 23:12:23 I [helper_pandas:16] - OpenVINO INT8 내보내기 완료: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\best_int8_openvino_model
2025-12-07 23:12:23 I [helper_pandas:21] - 생성된 파일: ['best.bin', 'best.xml', 'metadata.yaml']
2025-12-07 23:12:23 I [helper_pandas:

In [13]:
#!pip uninstall openvino openvino-dev
#!pip install openvino==2024.0.0 openvino-dev==2024.0.0

In [14]:
# ============================================================================
# 5. ONNX FP32 모델을 INT8로 양자화 (QDQ 형식) - 실제 데이터 기반
# ============================================================================
logger.info("="*80)
logger.info("ONNX Runtime Static Quantization (INT8 - QDQ 형식)")
logger.info("="*80)
logger.info("ONNX FP32 모델을 INT8 QDQ 형식으로 양자화합니다.")
logger.info("QDQ (Quantize-Dequantize): CPU에서 실행 가능한 형식")
logger.info("실제 데이터셋 이미지로 Calibration 수행 (랜덤 노이즈 X)")
logger.info("")

# 1. ONNX Runtime 설치 확인
logger.info("Step 1: ONNX Runtime 설치 확인")
install_if_missing("onnxruntime")

from onnxruntime.quantization import quantize_static, QuantFormat, QuantType
from onnxruntime.quantization.calibrate import CalibrationDataReader
import onnx
import numpy as np
import cv2

logger.info("  ✓ ONNX Runtime 로드 완료")

# 2. 데이터셋 재생성 (1000개씩)
logger.info("\nStep 2: 데이터셋 재생성 (Calibration용)")
logger.setLevel(logging.INFO)
yolo_dataset_path = my_cache_path("yolo", "the-oxfordiiit-pet-dataset")
yaml_path, train_df, valid_df, test_df, validation_results = oxfordiit_pet_to_yolo(
    max_samples_per_split=(1000, 1000, 1000),
    label_mode="species",
    output_dir=Path(yolo_dataset_path)
)
logger.setLevel(logging.DEBUG)

logger.info(f"  ✓ 데이터셋 경로: {yolo_dataset_path}")
logger.info(f"  ✓ Train: {len(train_df)}개, Valid: {len(valid_df)}개, Test: {len(test_df)}개")


# 3. 실제 데이터 기반 Calibration Reader 정의
class RealDataCalibrationReader(CalibrationDataReader):
    """실제 데이터셋 이미지로 Calibration 수행 (YOLOv8 표준 전처리 적용)"""
    
    def __init__(self, dataset_path, num_samples=300):
        self.dataset_path = dataset_path
        self.num_samples = num_samples
        self.image_files = self._load_image_list()
        self.current = 0
        logger.info(f"  ✓ RealDataCalibrationReader 초기화: {len(self.image_files)}개 이미지 로드")
    
    def _load_image_list(self):
        """train + valid 이미지 수집 (test는 제외)"""
        images = []
        for split in ['train', 'valid']:
            split_path = os.path.join(self.dataset_path, 'images', split)
            if os.path.exists(split_path):
                imgs = [f for f in os.listdir(split_path)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                images.extend([os.path.join(split_path, img) for img in imgs])
        
        # num_samples만큼 샘플링
        if len(images) > self.num_samples:
            images = images[:self.num_samples]
        
        logger.info(f"  ✓ Calibration 이미지 수집: {len(images)}개")
        return images
    
    def _preprocess_image(self, img_path):
        """YOLOv8 표준 전처리 (LetterBox + BGR2RGB + 정규화)"""
        try:
            from ultralytics.data.augment import LetterBox
            
            # 이미지 로드
            img = cv2.imread(img_path)
            if img is None:
                raise ValueError(f"이미지 로드 실패: {img_path}")
            
            # LetterBox 리사이즈 (auto=False로 정확히 640x640 보장)
            letterbox = LetterBox(640, auto=True, stride=32)
            img = letterbox(image=img)
            
            # BGR → RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # 정규화 [0, 1]
            img = img.astype(np.float32) / 255.0
            
            # HWC → CHW
            img = img.transpose(2, 0, 1)
            
            # 배치 차원 추가
            img_batch = img[np.newaxis, :]
            
            # 디버깅: 처음 3개 이미지만 크기 확인
            if self.current < 3:
                logger.info(f"  [이미지 {self.current}] {os.path.basename(img_path)}: 최종 크기={img_batch.shape}")
            
            # 크기 검증
            if img_batch.shape != (1, 3, 640, 640):
                raise ValueError(f"전처리 후 크기 불일치: {img_batch.shape} != (1, 3, 640, 640)")
            
            return img_batch
            
        except Exception as e:
            logger.error(f"이미지 전처리 실패 ({img_path}): {e}")
            raise
    
    def get_next(self):
        """다음 calibration 데이터 반환"""
        if self.current >= len(self.image_files):
            return None
        
        img_path = self.image_files[self.current]
        self.current += 1
        
        # 진행 상황 표시 (10%마다)
        if self.current % max(1, len(self.image_files) // 10) == 0:
            progress = (self.current / len(self.image_files)) * 100
            logger.info(f"  Calibration 진행: {self.current}/{len(self.image_files)} ({progress:.0f}%)")
        
        try:
            img_tensor = self._preprocess_image(img_path)
            return {'images': img_tensor}
        except Exception as e:
            logger.warning(f"이미지 로드 실패 ({img_path}): {e}")
            # 다음 이미지 시도
            return self.get_next()
    
    def rewind(self):
        """리더 초기화"""
        self.current = 0


# 4. 입력 ONNX FP32 모델 경로 확인
logger.info("\nStep 3: ONNX FP32 모델 경로 확인")

if 'onnx_fp32_final' in dir() and os.path.exists(onnx_fp32_final):
    onnx_fp32_path = onnx_fp32_final
elif 'onnx_fp32_export' in dir() and os.path.exists(onnx_fp32_export):
    onnx_fp32_path = onnx_fp32_export
else:
    onnx_fp32_path = os.path.join(yolov8m_path, 'weights', 'best.onnx')

if not os.path.exists(onnx_fp32_path):
    raise FileNotFoundError(f"ONNX FP32 모델을 찾을 수 없습니다: {onnx_fp32_path}")

fp32_size = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
logger.info(f"  ✓ ONNX FP32 모델 크기: {fp32_size:.2f} MB")
logger.info(f"  ✓ 모델 경로: {onnx_fp32_path}")


# 5. 출력 경로 설정
logger.info("\nStep 4: 출력 경로 설정")
output_onnx_int8_path = my_driver_path(yolov8m_path, 'weights', 'out_onnx_int8_qdq', create=True)
onnx_int8_final = os.path.join(output_onnx_int8_path, 'yolov8m_int8_qdq.onnx')
logger.info(f"  ✓ INT8 QDQ 모델 저장 경로: {onnx_int8_final}")


# 6. Calibration Reader 생성
logger.info("\nStep 5: Calibration Reader 생성")
calibration_reader = RealDataCalibrationReader(
    dataset_path=yolo_dataset_path,
    num_samples=500  # 300-500개 권장
)


# 7. Static Quantization with QDQ format
logger.info("\nStep 6: Static Quantization (QDQ 형식) 수행 중...")
logger.info("  - 실제 데이터셋 이미지로 Calibration")
logger.info("  - YOLOv8 표준 전처리 적용 (LetterBox auto=False → 정확히 640x640)")
logger.info("  - QDQ 노드 삽입 (CPU 실행 가능)")
logger.info("  - 처리 중... (약 2-5분 소요)")
logger.info("")

quantize_static(
    model_input=onnx_fp32_path,
    model_output=onnx_int8_final,
    calibration_data_reader=calibration_reader,
    quant_format=QuantFormat.QDQ,  # QDQ 형식 (CPU 호환)
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
)

logger.info("\n  ✓ Static Quantization (QDQ) 완료!")


# 8. 결과 확인
int8_size = os.path.getsize(onnx_int8_final) / (1024 * 1024)
compression_ratio = (1 - int8_size / fp32_size) * 100

logger.info("\n" + "="*80)
logger.info("양자화 결과")
logger.info("="*80)
logger.info(f"  - FP32 모델 크기: {fp32_size:.2f} MB")
logger.info(f"  - INT8 QDQ 모델 크기: {int8_size:.2f} MB")
logger.info(f"  - 압축률: {compression_ratio:.1f}%")
logger.info(f"  - Calibration 방식: 실제 데이터셋 이미지 ({len(calibration_reader.image_files)}개)")
logger.info(f"  - 전처리: YOLOv8 표준 (LetterBox auto=False → 640x640 + BGR2RGB + [0,1] 정규화)")
logger.info("")
logger.info(f"  - FP32 경로: {onnx_fp32_path}")
logger.info(f"  - INT8 QDQ 경로: {onnx_int8_final}")
logger.info("="*80)

2025-12-07 23:12:23 I [helper_pandas:4] - ================================================================================
2025-12-07 23:12:23 I [helper_pandas:5] - ONNX Runtime Static Quantization (INT8 - QDQ 형식)
2025-12-07 23:12:23 I [helper_pandas:6] - ================================================================================
2025-12-07 23:12:23 I [helper_pandas:7] - ONNX FP32 모델을 INT8 QDQ 형식으로 양자화합니다.
2025-12-07 23:12:23 I [helper_pandas:8] - QDQ (Quantize-Dequantize): CPU에서 실행 가능한 형식
2025-12-07 23:12:23 I [helper_pandas:9] - 실제 데이터셋 이미지로 Calibration 수행 (랜덤 노이즈 X)
2025-12-07 23:12:23 I [helper_pandas:10] - 
2025-12-07 23:12:23 I [helper_pandas:13] - Step 1: ONNX Runtime 설치 확인
onnxruntime 이미 설치되어 있음.
2025-12-07 23:12:23 I [helper_pandas:22] -   ✓ ONNX Runtime 로드 완료
2025-12-07 23:12:23 I [helper_pandas:25] - 
Step 2: 데이터셋 재생성 (Calibration용)


Processing test: 100%|██████████| 368/368 [00:01<00:00, 256.99it/s]
2025-12-07 23:12:38,327 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Abyssinian_104.txt
2025-12-07 23:12:39,918 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Bengal_111.txt
2025-12-07 23:12:41,122 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Bengal_175.txt
2025-12-07 23:12:44,991 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Egyptian_Mau_14.txt
2025-12-07 23:12:45,147 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\labels\train\Egyptian_Mau_156.txt
2025-12-07 23:12:45,434 - yolo_eval.oxfordiiit_pet_dataset - WARNING - 라벨 파일 없음: D:\temp\cache_local\yolo\the-ox

2025-12-07 23:12:58 I [helper_pandas:35] -   ✓ 데이터셋 경로: D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset
2025-12-07 23:12:58 I [helper_pandas:36] -   ✓ Train: 994개, Valid: 734개, Test: 368개
2025-12-07 23:12:58 I [helper_pandas:134] - 
Step 3: ONNX FP32 모델 경로 확인
2025-12-07 23:12:58 I [helper_pandas:147] -   ✓ ONNX FP32 모델 크기: 98.72 MB
2025-12-07 23:12:58 I [helper_pandas:148] -   ✓ 모델 경로: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx
2025-12-07 23:12:58 I [helper_pandas:152] - 
Step 4: 출력 경로 설정
2025-12-07 23:12:58 I [helper_utils_colab:384] - Created directory: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq
2025-12-07 23:12:58 I [helper_pandas:155] -   ✓ INT8 QDQ 모델 저장 경로: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_int8_qdq\yolov8m_int8_qdq.onnx
2025-12-07 23:12:58 I [helper_pandas:159] - 
Step 5: Calibration Reader 생성
2025-12-07 23:12:58 I [helper_pa

2025-12-07 23:13:01,192 - root - WARNING - Please consider to run pre-processing before quantization. Refer to example: https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md 


2025-12-07 23:13:04 I [helper_pandas:95] -   [이미지 1] Abyssinian_1.jpg: 최종 크기=(1, 3, 448, 640)
2025-12-07 23:13:04 E [helper_pandas:104] - 이미지 전처리 실패 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\train\Abyssinian_1.jpg): 전처리 후 크기 불일치: (1, 3, 448, 640) != (1, 3, 640, 640)
2025-12-07 23:13:04 W [helper_pandas:124] - 이미지 로드 실패 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\train\Abyssinian_1.jpg): 전처리 후 크기 불일치: (1, 3, 448, 640) != (1, 3, 640, 640)
2025-12-07 23:13:04 I [helper_pandas:95] -   [이미지 2] Abyssinian_10.jpg: 최종 크기=(1, 3, 640, 480)
2025-12-07 23:13:04 E [helper_pandas:104] - 이미지 전처리 실패 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\train\Abyssinian_10.jpg): 전처리 후 크기 불일치: (1, 3, 640, 480) != (1, 3, 640, 640)
2025-12-07 23:13:04 W [helper_pandas:124] - 이미지 로드 실패 (D:\temp\cache_local\yolo\the-oxfordiiit-pet-dataset\images\train\Abyssinian_10.jpg): 전처리 후 크기 불일치: (1, 3, 640, 480) != (1, 3, 640, 640)
2025-12-07 23:13:04 E [helper_pandas:104] - 이미지

2025-12-07 23:14:00,385 - root - WARNING - Please consider pre-processing before quantization. See https://github.com/microsoft/onnxruntime-inference-examples/blob/main/quantization/image_classification/cpu/ReadMe.md 


2025-12-07 23:14:00 I [helper_pandas:183] - 
  ✓ Static Quantization (QDQ) 완료!
2025-12-07 23:14:00 I [helper_pandas:190] - 
2025-12-07 23:14:00 I [helper_pandas:191] - 양자화 결과
2025-12-07 23:14:00 I [helper_pandas:192] - ================================================================================
2025-12-07 23:14:00 I [helper_pandas:193] -   - FP32 모델 크기: 98.72 MB
2025-12-07 23:14:00 I [helper_pandas:194] -   - INT8 QDQ 모델 크기: 25.10 MB
2025-12-07 23:14:00 I [helper_pandas:195] -   - 압축률: 74.6%
2025-12-07 23:14:00 I [helper_pandas:196] -   - Calibration 방식: 실제 데이터셋 이미지 (500개)
2025-12-07 23:14:00 I [helper_pandas:197] -   - 전처리: YOLOv8 표준 (LetterBox auto=False → 640x640 + BGR2RGB + [0,1] 정규화)
2025-12-07 23:14:00 I [helper_pandas:198] - 
2025-12-07 23:14:00 I [helper_pandas:199] -   - FP32 경로: D:\GoogleDrive\modeling\model\modeling16\yolov8m_20251205_2005\weights\out_onnx_fp32\yolov8m_fp32.onnx
2025-12-07 23:14:00 I [helper_pandas:200] -   - INT8 QDQ 경로: D:\GoogleDrive\modeling\model\mo